In [ ]:
from pathlib import Path
import os
import shutil


In [4]:
from IPython import get_ipython
from IPython.core.magic import register_cell_magic

ipython = get_ipython()


@register_cell_magic
def pybash(line, cell):
    cell_replaced = eval("f" + repr(cell))
    # print("Evaluating:\n{}\n-----------".format(cell_replaced))
    ipython.run_cell_magic('bash', '', cell_replaced)

In [ ]:
project_dir = Path(os.getcwd()).parent.parent
install_dir = project_dir / "install"
log_dir = project_dir / "logs" / "ior"
data_dir = project_dir / "data" / "ior"
output_dir = project_dir / "output" / "ior"
print("Directories created:")
for name, path in [("Install Directory", install_dir), 
                   ("Log Directory", log_dir), 
                   ("Data Directory", data_dir), 
                   ("Output Directory", output_dir)]:
    print(f"{name}: {path}")

Directories created:
Install Directory: /usr/WS2/haridev/dftracer-demo/install
Log Directory: /usr/WS2/haridev/dftracer-demo/logs/ior
Data Directory: /usr/WS2/haridev/dftracer-demo/data/ior
Output Directory: /usr/WS2/haridev/dftracer-demo/output/ior
Cleaned and created fresh folders for log, data, and output.


In [10]:

for dir_path in [log_dir, data_dir, output_dir]:
    if dir_path.exists():
        for item in dir_path.iterdir():
            if item.is_file():
                item.unlink()
            elif item.is_dir():
                shutil.rmtree(item)
    dir_path.mkdir(parents=True, exist_ok=True)
print("Cleaned and created fresh folders for log, data, and output.")

Cleaned and created fresh folders for log, data, and output.


In [11]:
import os

import importlib.util

spec = importlib.util.find_spec("dftracer")
if spec and spec.origin:
    dftracer_folder = os.path.dirname(spec.origin)
    print("dftracer folder:", dftracer_folder)
else:
    print("dftracer module not found.")

dftracer folder: /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dftracer


In [12]:
%%pybash
# DFTracer environment variables:
echo "Configuring DFTracer"

# DFTRACER_INC_METADATA: Include or exclude metadata (default 0)
export DFTRACER_INC_METADATA=1

# DFTRACER_INIT: DFTracer Mode FUNCTION/PRELOAD (default FUNCTION). For Hybrid use PRELOAD mode.
export DFTRACER_INIT=PRELOAD

# DFTRACER_DATA_DIR: Colon separated paths that will be traced for I/O accesses by profiler. 
# For tracing all directories use the string “all” (not recommended). 
export DFTRACER_DATA_DIR={data_dir}


# DFTRACER_ENABLE: Enable or Disable DFTracer (default 0).
export DFTRACER_ENABLE=1

# Setting path to DFTRACER preload so
DFTRACER_PRELOAD_SO={dftracer_folder}/lib64/libdftracer_preload.so


echo "Creating DFTRACER_DATA_DIR: $DFTRACER_DATA_DIR"
mkdir -p ${{DFTRACER_DATA_DIR}}


# DFTRACER_LOG_FILE: PATH To log file. In this case process id and app name is appended to file.
echo "Setting DFTRACER_LOG_FILE=${log_dir}/ior-sim"
export DFTRACER_LOG_FILE={log_dir}/ior-sim

echo "Removing previous logs ${{DFTRACER_LOG_FILE}}*"
rm -rf ${{DFTRACER_LOG_FILE}}*

rm -rf {output_dir}/*

echo "Running IOR with DFTracer"
flux run -n 2 -q pdebug --env LD_PRELOAD=${{DFTRACER_PRELOAD_SO}} {install_dir}/bin/ior -o=${{DFTRACER_DATA_DIR}}/test-sim.bat -b=32m -t=1m -O summaryFormat=CSV -O summaryFile={output_dir}/case1.csv -i 5 -F -w -m > {output_dir}/ior-sim.log 2>&1
echo "Result of Run"
cat {output_dir}/ior-sim.log
cat {output_dir}/case1.csv

Configuring DFTracer
Creating DFTRACER_DATA_DIR: /usr/WS2/haridev/dftracer-demo/data/ior
Setting DFTRACER_LOG_FILE=$/usr/WS2/haridev/dftracer-demo/logs/ior/ior-sim
Removing previous logs /usr/WS2/haridev/dftracer-demo/logs/ior/ior-sim*
Running IOR with DFTracer


In [13]:
import glob

pfw_files = glob.glob(str(log_dir / "*.pfw.gz"))
if pfw_files:
    print("Found .pfw.gz files:")
    for f in pfw_files:
        print(f)
else:
    print("No .pfw.gz files found in", log_dir)

Found .pfw.gz files:
/usr/WS2/haridev/dftracer-demo/logs/ior/ior-sim-9ea1e9bb92274ebf-preload.pfw.gz
/usr/WS2/haridev/dftracer-demo/logs/ior/ior-sim-e56beebf8d49b9a6-preload.pfw.gz


In [14]:
!gzip -dc {log_dir}/ior-sim* | (head -n 10; echo "..."; tail -n 5)

[
{"id":1,"name":"HH","cat":"dftracer","pid":87364,"tid":87364,"ph":"M","args":{"hhash":"20e24dec90f71cf8","name":"tuolumne1027","value":"20e24dec90f71cf8"}}
{"id":2,"name":"thread_name","cat":"dftracer","pid":87364,"tid":87364,"ph":"M","args":{"hhash":"20e24dec90f71cf8","name":"87364","value":"thread_name"}}
{"id":3,"name":"SH","cat":"dftracer","pid":87364,"tid":87364,"ph":"M","args":{"hhash":"20e24dec90f71cf8","name":"/usr/WS2/haridev/dftracer-demo/install/bin/ior;-o=/usr/WS2/haridev/dftracer-demo/data/ior/test-sim.bat;-b=32m;-t=1m;-O;summaryFormat=CSV;-O;summaryFile=/usr/WS2/haridev/dftracer-demo/output/ior/case1.csv;-i;5;-F;-w;-m","value":"1477ca3dbfc5e14b"}}
{"id":4,"name":"SH","cat":"dftracer","pid":87364,"tid":87364,"ph":"M","args":{"hhash":"20e24dec90f71cf8","name":"ior","value":"a01da183974134df"}}
{"id":5,"name":"start","cat":"dftracer","pid":87364,"tid":87364,"ts":1754810391071767,"dur":0,"ph":"X","args":{"hhash":"20e24dec90f71cf8","p_idx":-1,"level":1,"ppid":87363,"date":"S

In [15]:
from dfanalyzer import init_with_hydra
dfa = init_with_hydra(
    hydra_overrides=[
        f"trace_path={log_dir}",
    ]
)

/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 37939 instead
  warnings.warn(


In [16]:
dfa.client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:37939/status,
Dashboard: http://127.0.0.1:37939/status,Workers: 12
Total threads: 96,Total memory: 0 B
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43737,Workers: 12
Dashboard: http://127.0.0.1:37939/status,Total threads: 96
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:46035,Total threads: 8
Dashboard: http://127.0.0.1:40819/status,Memory: 0 B
Nanny: tcp://127.0.0.1:43711,


In [17]:
res = dfa.analyze_trace()
dfa.output.handle_result(res)

/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dask_expr/_collection.py:5063: FutureWarning: from_legacy_dataframe is deprecated and will be removed in a future release. The legacy implementation as a whole is deprecated and will be removed, making this method unnecessary.
  warnings.warn(


                                                Time Period Summary                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric                                                          ┃ Unit                  ┃                 Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ Job Time                                                        │ seconds               │                 0.700 │
│ Total Count                                                     │ count                 │                   702 │
│ Total Files                                                     │ count                 │                     0 │
│ Total Nodes                                                     │ count                 │                     0 │
│ Total Processes                                                 │ count                 │                     2 │
│ POSIX Count                                                     │ count                 │                   702 │
│ POSIX Size                                                      │ MB                    │               320.000 │
│ POSIX Bandwidth                                                 │ MB/s                  │               484.458 │
│ POSIX Avg Transfer Size                                         │ MB                    │                 0.456 │
└─────────────────────────────────────────────────────────────────┴───────────────────────┴───────────────────────┘
                                                  Layer Breakdown                                                  
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer       ┃         Time (s) ┃     Ops ┃          Ops/sec ┃         Size (MB) ┃              Bandwidth (MB/s) ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ POSIX       │            0.661 │     702 │         1062.780 │           320.000 │                       484.458 │
└─────────────┴──────────────────┴─────────┴──────────────────┴───────────────────┴───────────────────────────────┘